# ⚔️ CRUSADER — F02 CASTELLAN
## Config Créative + Viewer → roadmap.json

> *"The Castellan holds the line, the Castellan shapes the battle."*

---

**Ce notebook tourne sur CPU — aucun GPU requis.**

### Étapes :
1. Montage Google Drive
2. Installation Flask
3. Téléchargement des scripts depuis GitHub
4. Configuration des chemins
5. Validation CUSTOS check-out
6. Lancement du serveur Flask
7. Ouverture du viewer (configuration opérateur)
8. Validation CUSTOS check-in
9. Aperçu du roadmap.json produit

---
## Étape 1 — Montage Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive monté sur /content/drive')

---
## Étape 2 — Installation Flask

In [ ]:
import subprocess, sys

print('Installation de Flask...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'flask>=2.0', '-q'],
    capture_output=True, text=True
)
if result.returncode == 0:
    import flask
    print(f'[OK] Flask {flask.__version__} installé.')
else:
    print('[ERREUR]', result.stderr)

---
## Étape 3 — Téléchargement des scripts depuis GitHub

In [ ]:
import urllib.request, os

REPO_RAW = 'https://raw.githubusercontent.com/kioka8877-ux/CRUSADER/main'
SCRIPT_DIR = '/content/crusader_f02'
os.makedirs(SCRIPT_DIR, exist_ok=True)

files_to_download = [
    'F02_CASTELLAN/CODEBASE/crs_f02_castellan.py',
    'F02_CASTELLAN/CODEBASE/crs_f02_viewer.html',
    'CRS_CUSTOS.py',
]

for rel_path in files_to_download:
    url       = f'{REPO_RAW}/{rel_path}'
    dest_name = os.path.basename(rel_path)
    dest_path = os.path.join(SCRIPT_DIR, dest_name)
    urllib.request.urlretrieve(url, dest_path)
    print(f'[OK] {dest_name} téléchargé → {dest_path}')

print('\nTous les scripts sont prêts.')

---
## Étape 4 — Configuration des chemins

> **Modifiez `DRIVE_BASE` si votre structure Google Drive est différente.**

In [ ]:
import os

# ── MODIFIEZ ICI SI NÉCESSAIRE ─────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_CRUSADER'
# ──────────────────────────────────────────────────────────────────────────────

F02_IN  = os.path.join(DRIVE_BASE, 'F02_CASTELLAN', 'IN')
F02_OUT = os.path.join(DRIVE_BASE, 'F02_CASTELLAN', 'OUT')

SCRIPT_DIR = '/content/crusader_f02'
FLASK_PORT = 5002

print('Configuration :')
print(f'  F02 IN  : {F02_IN}')
print(f'  F02 OUT : {F02_OUT}')
print(f'  Port    : {FLASK_PORT}')
print()

# Vérification rapide
timing_ok = os.path.isfile(os.path.join(F02_IN, 'timing.json'))
images_ok = os.path.isdir(os.path.join(F02_IN, 'images'))
print(f'  timing.json présent : {"✓" if timing_ok else "✗ MANQUANT"}')
print(f'  images/     présent : {"✓" if images_ok else "✗ MANQUANT"}')

---
## Étape 5 — Validation CUSTOS check-out (F02)

In [ ]:
import subprocess, sys

custos_path = os.path.join(SCRIPT_DIR, 'CRS_CUSTOS.py')

result = subprocess.run(
    [sys.executable, custos_path,
     '--frigate', 'F02',
     '--mode', 'check-out',
     '--drive-base', DRIVE_BASE],
    capture_output=False, text=True
)

if result.returncode != 0:
    print('\n[STOP] CUSTOS check-out FAIL. Vérifiez que timing.json et images/ sont en place dans F02/IN/.')
else:
    print('\n[OK] Check-out validé. Vous pouvez lancer le serveur.')

---
## Étape 6 — Lancement du serveur Flask

> Le serveur tourne en arrière-plan dans Colab. L'URL publique s'affiche ci-dessous.

In [ ]:
import threading, subprocess, sys, time, os
from google.colab.output import eval_js

castellan_script = os.path.join(SCRIPT_DIR, 'crs_f02_castellan.py')
viewer_html      = os.path.join(SCRIPT_DIR, 'crs_f02_viewer.html')

# Lancement Flask en sous-processus
flask_proc = subprocess.Popen(
    [sys.executable, castellan_script,
     '--input',  F02_IN,
     '--output', F02_OUT,
     '--viewer', viewer_html,
     '--port',   str(FLASK_PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Attente démarrage
time.sleep(2)

if flask_proc.poll() is not None:
    print('[ERREUR] Le serveur Flask a planté. Vérifiez les logs.')
else:
    print(f'[OK] Serveur Flask actif sur le port {FLASK_PORT}.')
    print('\nLe viewer est accessible à :')
    print(eval_js(f'google.colab.kernel.proxyPort({FLASK_PORT})'))

---
## Étape 7 — Viewer interactif

**Ouvrez l'URL ci-dessus dans un nouvel onglet**, puis :
1. Assignez une image à chaque segment (ou utilisez "Auto-assigner")
2. Configurez le style (format, polices, couleurs, effets)
3. Cliquez **VALIDER & SAUVEGARDER**
4. Attendez la confirmation verte, puis revenez ici pour continuer.

> Une fois `roadmap.json` sauvegardé dans `F02/OUT/`, passez à l'étape 8.

---
## Étape 8 — Validation CUSTOS check-in (F02)

In [ ]:
import subprocess, sys, os

custos_path = os.path.join(SCRIPT_DIR, 'CRS_CUSTOS.py')

result = subprocess.run(
    [sys.executable, custos_path,
     '--frigate', 'F02',
     '--mode', 'check-in',
     '--drive-base', DRIVE_BASE],
    capture_output=False, text=True
)

if result.returncode != 0:
    print('\n[STOP] CUSTOS check-in FAIL. Retournez dans le viewer et validez le roadmap.json.')
else:
    print('\n[OK] Check-in validé. roadmap.json prêt pour transfert vers F02 → F03.')

# Arrêt propre du serveur Flask
if 'flask_proc' in dir() and flask_proc.poll() is None:
    flask_proc.terminate()
    print('[OK] Serveur Flask arrêté.')

---
## Étape 9 — Aperçu du roadmap.json produit

In [ ]:
import json, os

roadmap_path = os.path.join(F02_OUT, 'roadmap.json')

if not os.path.isfile(roadmap_path):
    print(f'[ERREUR] roadmap.json introuvable : {roadmap_path}')
else:
    with open(roadmap_path, 'r', encoding='utf-8') as f:
        rm = json.load(f)

    print('══════════════════════════════════════════')
    print('  F02 CASTELLAN — MISSION ACCOMPLIE')
    print('══════════════════════════════════════════')
    print(f'  Format    : {rm["meta"]["format"]} ({rm["meta"]["width"]}×{rm["meta"]["height"]})')
    print(f'  FPS       : {rm["meta"]["fps"]}')
    print(f'  Segments  : {len(rm["timeline"])}')
    print(f'  Validé    : {rm["validated_by_magos"]}')
    print()
    print('  Style :')
    for k, v in rm['style'].items():
        print(f'    {k}: {v}')
    print()
    print('  Timeline (5 premiers segments) :')
    for item in rm['timeline'][:5]:
        img = item.get('image_file') or '(aucune)'
        print(f'    [{item["id"]}] f{item["start_frame"]}→f{item["end_frame"]} | img:{img} | "{item["text_subtitles"][:40]}"')
    print('══════════════════════════════════════════')